# Personalized Movie Recommendation System — Full ML Notebook

**Dataset:** MovieLens 32M-scale data  
**Core method:** Collaborative Filtering + Matrix Factorization  
**Additional components:** Popularity baseline, Item-Based CF, TF-IDF content model, Hybrid Recommendation, Cold Start, Explainability, Diversity, Evaluation, Model Persistence.

Run the notebook top-to-bottom. It is designed for a large ratings file (~32M rows) and uses sparse matrices and memory-conscious dtypes.


## Recommended project structure

```text
movie-recommender/
├── data/
│   └── ml-32m/
│       ├── movies.csv
│       ├── ratings.csv
│       ├── links.csv
│       └── tags.csv
├── notebooks/
│   └── MovieLens_Recommendation_System_Full_ML_Notebook.ipynb
├── models/
├── outputs/
└── README.md
```

Change `DATA_DIR` below if your CSV files are elsewhere.


In [ ]:
# CELL 1 — Install dependencies
%pip install -q pandas numpy scipy scikit-learn matplotlib seaborn joblib tqdm implicit pyarrow


In [ ]:
# CELL 2 — Imports
from pathlib import Path
import json
import gc
import time
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from implicit.als import AlternatingLeastSquares
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

print("Environment ready.")


In [ ]:
# CELL 3 — Paths and configuration
DATA_DIR = Path("ml-32m")  # <-- CHANGE THIS

MOVIES_PATH = DATA_DIR / "movies.csv"
RATINGS_PATH = DATA_DIR / "ratings.csv"
LINKS_PATH = DATA_DIR / "links.csv"
TAGS_PATH = DATA_DIR / "tags.csv"

MODEL_DIR = Path("models")
OUTPUT_DIR = Path("outputs")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for p in [MOVIES_PATH, RATINGS_PATH, LINKS_PATH, TAGS_PATH]:
    print(f"{p}: {'FOUND' if p.exists() else 'NOT FOUND'}")


## 1. Load the data

The ratings file is the large file. We explicitly select columns and smaller dtypes. Tags are loaded later because they are not required for the first collaborative-filtering model.

In [ ]:
# CELL 4 — Load movies
movies = pd.read_csv(
    MOVIES_PATH,
    dtype={
        "movieId": "int32",
        "title": "string",
        "genres": "string"
    }
)

print("movies.shape:", movies.shape)
display(movies.head())


In [ ]:
# CELL 5 — Load ratings efficiently
ratings = pd.read_csv(
    RATINGS_PATH,
    usecols=["userId", "movieId", "rating", "timestamp"],
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32",
        "timestamp": "int32"
    },
    memory_map=True
)

print("ratings.shape:", ratings.shape)
ratings.info(memory_usage="deep")
display(ratings.head())


In [ ]:
# CELL 6 — Load links
links = pd.read_csv(
    LINKS_PATH,
    dtype={
        "movieId": "int32",
        "imdbId": "Int32",
        "tmdbId": "Int32"
    }
)

print("links.shape:", links.shape)
display(links.head())


In [ ]:
# CELL 7 — Dataset summary
print(f"Movies: {len(movies):,}")
print(f"Ratings: {len(ratings):,}")
print(f"Unique users: {ratings['userId'].nunique():,}")
print(f"Unique rated movies: {ratings['movieId'].nunique():,}")
print(f"Rating range: {ratings['rating'].min()} to {ratings['rating'].max()}")

print("\nMissing values — movies")
display(movies.isna().sum().to_frame("missing"))

print("\nMissing values — ratings")
display(ratings.isna().sum().to_frame("missing"))


## 2. Validation and EDA

In [ ]:
# CELL 8 — Duplicate / basic validation
duplicate_count = ratings.duplicated(subset=["userId", "movieId"]).sum()
print("Duplicate user-movie pairs:", duplicate_count)

print("\nRating values:")
display(ratings["rating"].value_counts().sort_index().to_frame("count"))


In [ ]:
# CELL 9 — Rating distribution
rating_counts = ratings["rating"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(x=rating_counts.index.astype(str), y=rating_counts.values)
plt.title("MovieLens Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 10 — Ratings per user
user_activity = ratings.groupby("userId").size().sort_values(ascending=False)
print(user_activity.describe())

plt.figure(figsize=(10, 5))
plt.hist(user_activity.values, bins=100)
plt.yscale("log")
plt.title("Ratings per User")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Users (log scale)")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 11 — Ratings per movie
movie_activity = ratings.groupby("movieId").size().sort_values(ascending=False)
print(movie_activity.describe())

plt.figure(figsize=(10, 5))
plt.hist(movie_activity.values, bins=100)
plt.yscale("log")
plt.title("Ratings per Movie")
plt.xlabel("Number of Ratings")
plt.ylabel("Number of Movies (log scale)")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 12 — Most rated movies
most_rated = (
    ratings.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        average_rating=("rating", "mean")
    )
    .reset_index()
    .sort_values("rating_count", ascending=False)
    .merge(movies, on="movieId", how="left")
)
display(most_rated.head(20))


## 3. Chronological train / validation / test split

For each user with at least 3 interactions, the two most recent interactions are held out: the latest for test and the previous one for validation. This avoids training on future interactions.

In [ ]:
# CELL 13 — Chronological split
ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)

user_counts_all = ratings.groupby("userId")["movieId"].transform("size")
user_position_all = ratings.groupby("userId").cumcount()

eligible = user_counts_all >= 3

ratings_eligible = ratings.loc[eligible].copy()
counts_eligible = user_counts_all.loc[eligible]
positions_eligible = user_position_all.loc[eligible]

test_mask = positions_eligible == (counts_eligible - 1)
val_mask = positions_eligible == (counts_eligible - 2)
train_mask = positions_eligible < (counts_eligible - 2)

train = ratings_eligible.loc[train_mask].copy()
validation = ratings_eligible.loc[val_mask].copy()
test = ratings_eligible.loc[test_mask].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)


## 4. Filter sparse users and movies

Initial project defaults: at least 5 training ratings per user and at least 10 training ratings per movie. These are hyperparameters for experimentation, not universal constants.

In [ ]:
# CELL 14 — Sparse filtering
MIN_USER_TRAIN = 5
MIN_ITEM_TRAIN = 10

valid_users = train.groupby("userId").size()
valid_users = valid_users[valid_users >= MIN_USER_TRAIN].index

train = train[train["userId"].isin(valid_users)].copy()
validation = validation[validation["userId"].isin(valid_users)].copy()
test = test[test["userId"].isin(valid_users)].copy()

valid_movies = train.groupby("movieId").size()
valid_movies = valid_movies[valid_movies >= MIN_ITEM_TRAIN].index

train = train[train["movieId"].isin(valid_movies)].copy()
validation = validation[validation["movieId"].isin(valid_movies)].copy()
test = test[test["movieId"].isin(valid_movies)].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
print("Users:", train["userId"].nunique())
print("Movies:", train["movieId"].nunique())


In [ ]:
# CELL 15 — Check held-out catalogue overlap
train_movie_set = set(train["movieId"].unique())

print("Validation rows:", len(validation))
print("Validation movie overlap with train:",
      round(validation["movieId"].isin(train_movie_set).mean(), 4))

print("Test rows:", len(test))
print("Test movie overlap with train:",
      round(test["movieId"].isin(train_movie_set).mean(), 4))


## 5. Create contiguous user/movie indices

In [ ]:
# CELL 16 — Raw IDs to contiguous indices
unique_users = pd.Index(train["userId"].unique(), name="userId")
unique_movies = pd.Index(train["movieId"].unique(), name="movieId")

user_to_idx = {int(v): int(i) for i, v in enumerate(unique_users)}
movie_to_idx = {int(v): int(i) for i, v in enumerate(unique_movies)}

idx_to_user = {int(i): int(v) for i, v in enumerate(unique_users)}
idx_to_movie = {int(i): int(v) for i, v in enumerate(unique_movies)}

n_users = len(unique_users)
n_movies = len(unique_movies)

print("Users :", n_users)
print("Movies:", n_movies)


In [ ]:
# CELL 17 — Apply internal indices
def add_internal_ids(df):
    df = df.copy()
    df["user_idx"] = df["userId"].map(user_to_idx)
    df["movie_idx"] = df["movieId"].map(movie_to_idx)
    return df.dropna(subset=["user_idx", "movie_idx"])

train = add_internal_ids(train)
validation = add_internal_ids(validation)
test = add_internal_ids(test)

for df in [train, validation, test]:
    df["user_idx"] = df["user_idx"].astype("int32")
    df["movie_idx"] = df["movie_idx"].astype("int32")

display(train.head())


## 6. Sparse User × Movie matrix

Do not create a dense `user × movie` dataframe. With this dataset, that would waste enormous memory. CSR sparse matrices keep only observed interactions.

In [ ]:
# CELL 18 — Sparse rating matrix
user_movie_matrix = sp.csr_matrix(
    (
        train["rating"].to_numpy(dtype=np.float32),
        (
            train["user_idx"].to_numpy(dtype=np.int32),
            train["movie_idx"].to_numpy(dtype=np.int32)
        )
    ),
    shape=(n_users, n_movies),
    dtype=np.float32
)

print("Shape:", user_movie_matrix.shape)
print("Non-zero ratings:", f"{user_movie_matrix.nnz:,}")

total_cells = n_users * n_movies
sparsity = 1 - (user_movie_matrix.nnz / total_cells)

print("Possible cells:", f"{total_cells:,}")
print(f"Sparsity: {sparsity:.4%}")


## 7. Popularity baseline

In [ ]:
# CELL 19 — Train-only popularity baseline
MIN_POPULARITY_RATINGS = 50

popularity_train = (
    train.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        avg_rating=("rating", "mean")
    )
    .reset_index()
)

popular_movies_train = popularity_train[
    popularity_train["rating_count"] >= MIN_POPULARITY_RATINGS
].copy()

popular_movies_train["popularity_score"] = (
    popular_movies_train["avg_rating"]
    * np.log1p(popular_movies_train["rating_count"])
)

popular_movies_train = (
    popular_movies_train
    .sort_values("popularity_score", ascending=False)
    .merge(movies, on="movieId", how="left")
)

display(
    popular_movies_train[
        ["movieId", "title", "genres", "avg_rating", "rating_count", "popularity_score"]
    ].head(20)
)


In [ ]:
# CELL 20 — Popularity recommender
def popularity_recommendations(n=10):
    return (
        popular_movies_train[
            ["movieId", "title", "genres", "avg_rating", "rating_count", "popularity_score"]
        ]
        .head(n)
        .reset_index(drop=True)
    )

display(popularity_recommendations(10))


## 8. Positive-feedback matrix for implicit ALS

We treat ratings >= 4.0 as positive preference signals. The threshold can be tuned later.

In [ ]:
# CELL 21 — Positive interactions
POSITIVE_RATING_THRESHOLD = 4.0

positive_train = train[
    train["rating"] >= POSITIVE_RATING_THRESHOLD
].copy()

# Simple confidence: 4.0 -> 1.0 and 5.0 -> 2.0
positive_confidence = (
    1.0
    + (
        positive_train["rating"].to_numpy(dtype=np.float32)
        - POSITIVE_RATING_THRESHOLD
    )
).astype(np.float32)

user_items = sp.csr_matrix(
    (
        positive_confidence,
        (
            positive_train["user_idx"].to_numpy(dtype=np.int32),
            positive_train["movie_idx"].to_numpy(dtype=np.int32)
        )
    ),
    shape=(n_users, n_movies),
    dtype=np.float32
)

item_users = user_items.T.tocsr()

print("Positive interactions:", f"{len(positive_train):,}")
print("user_items:", user_items.shape, "nnz:", f"{user_items.nnz:,}")
print("item_users:", item_users.shape)


## 9. Train ALS matrix factorization

In [ ]:
# CELL 22 — Train ALS
ALS_FACTORS = 64
ALS_REGULARIZATION = 0.08
ALS_ITERATIONS = 20

start = time.time()

als_model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    regularization=ALS_REGULARIZATION,
    iterations=ALS_ITERATIONS,
    random_state=SEED
)

als_model.fit(item_users, show_progress=True)

print(f"Training time: {(time.time() - start)/60:.2f} minutes")
print("User factors:", als_model.user_factors.shape)
print("Item factors:", als_model.item_factors.shape)


In [ ]:
# CELL 23 — Personalized recommendations
def recommend_for_user(raw_user_id, n=10, filter_seen=True):
    raw_user_id = int(raw_user_id)

    if raw_user_id not in user_to_idx:
        raise ValueError(f"User {raw_user_id} is not in the training catalogue.")

    user_idx = user_to_idx[raw_user_id]

    item_indices, scores = als_model.recommend(
        userid=user_idx,
        user_items=user_items[user_idx],
        N=n,
        filter_already_liked_items=filter_seen
    )

    result = pd.DataFrame({
        "movie_idx": item_indices.astype(int),
        "score": scores.astype(float)
    })

    result["movieId"] = result["movie_idx"].map(idx_to_movie)
    result = result.merge(movies, on="movieId", how="left")

    return result[["movieId", "title", "genres", "score"]].reset_index(drop=True)


In [ ]:
# CELL 24 — Test ALS recommendations
example_user = int(train["userId"].value_counts().index[0])

print("Example user:", example_user)
display(recommend_for_user(example_user, 10))


In [ ]:
# CELL 25 — User history
def user_history(raw_user_id, n=20):
    return (
        train[train["userId"] == int(raw_user_id)]
        .sort_values("timestamp", ascending=False)
        .head(n)
        .merge(movies, on="movieId", how="left")
        [["movieId", "title", "genres", "rating", "timestamp"]]
        .reset_index(drop=True)
    )

display(user_history(example_user, 20))


## 10. Offline evaluation

In [ ]:
# CELL 26 — Test ground truth
test_eval = test.dropna(subset=["user_idx", "movie_idx"]).copy()
test_truth = (
    test_eval
    .groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)

print("Evaluation users:", len(test_truth))


In [ ]:
# CELL 27 — Evaluate top-K recommender
def evaluate_recommender(
    model,
    user_items_matrix,
    test_truth,
    k=10,
    max_users=10000,
    seed=42
):
    rng = np.random.default_rng(seed)

    users = np.array(sorted(test_truth.keys()), dtype=np.int32)

    if len(users) > max_users:
        users = rng.choice(users, size=max_users, replace=False)

    recommended, _ = model.recommend(
        userid=users,
        user_items=user_items_matrix[users],
        N=k,
        filter_already_liked_items=True
    )

    precision_sum = 0.0
    recall_sum = 0.0
    ndcg_sum = 0.0
    hit_sum = 0.0

    recommended_items = set()

    for row_idx, user_idx in enumerate(users):
        true_items = test_truth[int(user_idx)]

        hits = 0
        dcg = 0.0

        for rank, item_idx in enumerate(recommended[row_idx], start=1):
            item_idx = int(item_idx)
            recommended_items.add(item_idx)

            if item_idx in true_items:
                hits += 1
                dcg += 1 / np.log2(rank + 1)

        precision_sum += hits / k
        recall_sum += hits / max(len(true_items), 1)
        hit_sum += float(hits > 0)

        ideal_hits = min(len(true_items), k)
        if ideal_hits:
            idcg = sum(
                1 / np.log2(rank + 1)
                for rank in range(1, ideal_hits + 1)
            )
            ndcg_sum += dcg / idcg

    num_users = len(users)

    return {
        f"Precision@{k}": precision_sum / num_users,
        f"Recall@{k}": recall_sum / num_users,
        f"HitRate@{k}": hit_sum / num_users,
        f"NDCG@{k}": ndcg_sum / num_users,
        f"Coverage@{k}": len(recommended_items) / user_items_matrix.shape[1]
    }


In [ ]:
# CELL 28 — Evaluate ALS
als_metrics = evaluate_recommender(
    model=als_model,
    user_items_matrix=user_items,
    test_truth=test_truth,
    k=10,
    max_users=10000,
    seed=SEED
)

display(pd.DataFrame([als_metrics], index=["ALS"]))


In [ ]:
# CELL 29 — Evaluate popularity
popular_movie_ids = (
    popular_movies_train["movieId"]
    .astype(int)
    .tolist()
)

popular_movie_indices = np.array(
    [movie_to_idx[m] for m in popular_movie_ids if m in movie_to_idx],
    dtype=np.int32
)

def evaluate_popularity(
    popular_items,
    test_truth,
    k=10,
    max_users=10000,
    seed=42
):
    rng = np.random.default_rng(seed)
    users = np.array(sorted(test_truth.keys()), dtype=np.int32)

    if len(users) > max_users:
        users = rng.choice(users, size=max_users, replace=False)

    top_items = popular_items[:k]

    precision_sum = recall_sum = hit_sum = ndcg_sum = 0.0

    for user_idx in users:
        true_items = test_truth[int(user_idx)]
        hits = 0
        dcg = 0.0

        for rank, item_idx in enumerate(top_items, start=1):
            if int(item_idx) in true_items:
                hits += 1
                dcg += 1 / np.log2(rank + 1)

        precision_sum += hits / k
        recall_sum += hits / max(len(true_items), 1)
        hit_sum += float(hits > 0)

        ideal_hits = min(len(true_items), k)
        if ideal_hits:
            idcg = sum(
                1 / np.log2(rank + 1)
                for rank in range(1, ideal_hits + 1)
            )
            ndcg_sum += dcg / idcg

    n = len(users)

    return {
        f"Precision@{k}": precision_sum / n,
        f"Recall@{k}": recall_sum / n,
        f"HitRate@{k}": hit_sum / n,
        f"NDCG@{k}": ndcg_sum / n
    }

popularity_metrics = evaluate_popularity(
    popular_movie_indices,
    test_truth,
    k=10,
    max_users=10000,
    seed=SEED
)

display(pd.DataFrame([popularity_metrics], index=["Popularity"]))


In [ ]:
# CELL 30 — Compare baseline and ALS
comparison = pd.DataFrame([
    {"Model": "Popularity", **popularity_metrics},
    {"Model": "ALS Collaborative Filtering", **als_metrics}
])

display(comparison)

comparison.to_csv(
    OUTPUT_DIR / "baseline_vs_als.csv",
    index=False
)


## 11. Item-Based Collaborative Filtering

We use nearest-neighbor search on the sparse item × user matrix. We deliberately avoid materializing a full dense movie × movie similarity matrix.

In [ ]:
# CELL 31 — Fit item-item KNN
item_rating_matrix = user_movie_matrix.T.tocsr()

ITEM_KNN_NEIGHBORS = 20

item_knn = NearestNeighbors(
    n_neighbors=ITEM_KNN_NEIGHBORS,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

item_knn.fit(item_rating_matrix)
print("Item KNN fitted.")


In [ ]:
# CELL 32 — Item-based similar movies
def item_based_similar_movies(raw_movie_id, n=10):
    raw_movie_id = int(raw_movie_id)

    if raw_movie_id not in movie_to_idx:
        raise ValueError("Movie is not in the training catalogue.")

    movie_idx = movie_to_idx[raw_movie_id]

    distances, indices = item_knn.kneighbors(
        item_rating_matrix[movie_idx],
        n_neighbors=min(n + 1, ITEM_KNN_NEIGHBORS)
    )

    result = pd.DataFrame({
        "movie_idx": indices.flatten().astype(int),
        "distance": distances.flatten().astype(float)
    })

    result["similarity"] = 1.0 - result["distance"]
    result["movieId"] = result["movie_idx"].map(idx_to_movie)
    result = result[result["movieId"] != raw_movie_id]
    result = result.merge(movies, on="movieId", how="left")

    return result[
        ["movieId", "title", "genres", "similarity"]
    ].head(n).reset_index(drop=True)

liked_movie = int(
    train[
        (train["userId"] == example_user) &
        (train["rating"] >= 4.0)
    ]
    .sort_values(["rating", "timestamp"], ascending=False)
    .iloc[0]["movieId"]
)

display(movies[movies["movieId"] == liked_movie])
display(item_based_similar_movies(liked_movie, 10))


In [ ]:
# CELL 33 — ALS similar movies
def als_similar_movies(raw_movie_id, n=10):
    raw_movie_id = int(raw_movie_id)

    if raw_movie_id not in movie_to_idx:
        raise ValueError("Movie is not in the training catalogue.")

    movie_idx = movie_to_idx[raw_movie_id]

    indices, scores = als_model.similar_items(
        movie_idx,
        N=n + 1
    )

    result = pd.DataFrame({
        "movie_idx": indices.astype(int),
        "similarity": scores.astype(float)
    })

    result["movieId"] = result["movie_idx"].map(idx_to_movie)
    result = result[result["movieId"] != raw_movie_id]
    result = result.merge(movies, on="movieId", how="left")

    return result[
        ["movieId", "title", "genres", "similarity"]
    ].head(n).reset_index(drop=True)

display(als_similar_movies(liked_movie, 10))


In [ ]:
# CELL 34 — ALS similar users
def als_similar_users(raw_user_id, n=10):
    raw_user_id = int(raw_user_id)

    if raw_user_id not in user_to_idx:
        raise ValueError("User is not in the training catalogue.")

    user_idx = user_to_idx[raw_user_id]

    indices, scores = als_model.similar_users(
        user_idx,
        N=n + 1
    )

    result = pd.DataFrame({
        "user_idx": indices.astype(int),
        "similarity": scores.astype(float)
    })

    result["userId"] = result["user_idx"].map(idx_to_user)
    result = result[result["userId"] != raw_user_id]

    return result.head(n).reset_index(drop=True)

display(als_similar_users(example_user, 10))


## 12. User Taste Profile

In [ ]:
# CELL 35 — Taste profile
def user_taste_profile(raw_user_id, top_n=10):
    raw_user_id = int(raw_user_id)

    history = train[
        train["userId"] == raw_user_id
    ].merge(movies, on="movieId", how="left")

    if history.empty:
        return None

    genre_scores = {}

    for genres, rating in zip(history["genres"], history["rating"]):
        for genre in str(genres).split("|"):
            if genre in {"(no genres listed)", "nan"}:
                continue
            genre_scores[genre] = genre_scores.get(genre, 0.0) + float(rating)

    top_genres = (
        pd.DataFrame(
            list(genre_scores.items()),
            columns=["genre", "score"]
        )
        .sort_values("score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return {
        "userId": raw_user_id,
        "movies_rated": int(len(history)),
        "average_rating": float(history["rating"].mean()),
        "top_genres": top_genres
    }

profile = user_taste_profile(example_user)
print("Movies rated:", profile["movies_rated"])
print("Average rating:", round(profile["average_rating"], 3))
display(profile["top_genres"])


## 13. Explainability

This creates a human-readable product explanation from learned item-factor similarity. It is an explanation layer around the model, not a claim that ALS itself produces natural-language reasons.

In [ ]:
# CELL 36 — Explanation helpers
item_factors_normalized = normalize(
    als_model.item_factors.astype(np.float32),
    norm="l2",
    axis=1
)

def explanation_for_movie(raw_user_id, recommended_movie_id):
    raw_user_id = int(raw_user_id)
    recommended_movie_id = int(recommended_movie_id)

    if raw_user_id not in user_to_idx:
        return "User is not in the training catalogue."

    if recommended_movie_id not in movie_to_idx:
        return "Movie is not in the training catalogue."

    liked = (
        train[
            (train["userId"] == raw_user_id) &
            (train["rating"] >= 4.0)
        ]
        .sort_values(["rating", "timestamp"], ascending=False)
    )

    if liked.empty:
        return "Not enough positive history to generate an explanation."

    candidate_idx = movie_to_idx[recommended_movie_id]
    candidate_vector = item_factors_normalized[candidate_idx].reshape(1, -1)

    liked_indices = [
        movie_to_idx[mid]
        for mid in liked["movieId"].tolist()
        if mid in movie_to_idx
    ]

    similarities = (
        item_factors_normalized[liked_indices] @ candidate_vector.T
    ).ravel()

    source_movie_idx = liked_indices[int(np.argmax(similarities))]
    source_movie_id = idx_to_movie[source_movie_idx]

    source_title = movies.loc[
        movies["movieId"] == source_movie_id, "title"
    ].iloc[0]

    target_title = movies.loc[
        movies["movieId"] == recommended_movie_id, "title"
    ].iloc[0]

    return (
        f"Because you rated '{source_title}' highly, and the model "
        f"learned similar preference patterns for '{target_title}'."
    )


In [ ]:
# CELL 37 — Explainable recommendations
def explainable_recommendations(raw_user_id, n=10):
    result = recommend_for_user(raw_user_id, n=n).copy()
    result["explanation"] = [
        explanation_for_movie(raw_user_id, int(movie_id))
        for movie_id in result["movieId"]
    ]
    return result

display(explainable_recommendations(example_user, 10))


## 14. TF-IDF content model

In [ ]:
# CELL 38 — Movie text features
movie_metadata = movies.copy()

movie_metadata["genres_text"] = (
    movie_metadata["genres"]
    .fillna("")
    .str.replace("|", " ", regex=False)
)

movie_metadata["text"] = (
    movie_metadata["title"].fillna("")
    + " "
    + movie_metadata["genres_text"]
)

display(movie_metadata.head())


In [ ]:
# CELL 39 — Fit TF-IDF
TFIDF_MAX_FEATURES = 50000

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=TFIDF_MAX_FEATURES,
    min_df=2,
    ngram_range=(1, 2)
)

movie_tfidf = tfidf.fit_transform(movie_metadata["text"])

print("Full movie TF-IDF:", movie_tfidf.shape)


In [ ]:
# CELL 40 — Align TF-IDF to training movie order
metadata_lookup = movie_metadata.set_index("movieId")

training_text = (
    metadata_lookup
    .reindex(unique_movies)["text"]
    .fillna("")
)

training_tfidf = tfidf.transform(training_text)

print("Training TF-IDF:", training_tfidf.shape)


In [ ]:
# CELL 41 — Existing-user content profile
def build_user_content_profile(raw_user_id, min_rating=4.0):
    raw_user_id = int(raw_user_id)

    history = train[
        (train["userId"] == raw_user_id) &
        (train["rating"] >= min_rating)
    ].copy()

    if history.empty:
        return None

    movie_indices = [
        movie_to_idx[mid]
        for mid in history["movieId"].tolist()
        if mid in movie_to_idx
    ]

    if not movie_indices:
        return None

    ratings_for_profile = (
        history[history["movieId"].isin([idx_to_movie[i] for i in movie_indices])]
        .set_index("movieId")
        .loc[[idx_to_movie[i] for i in movie_indices], "rating"]
        .to_numpy(dtype=np.float32)
    )

    profile = (
        training_tfidf[movie_indices]
        .multiply(ratings_for_profile[:, None])
        .sum(axis=0)
    )

    return normalize(sp.csr_matrix(profile), norm="l2")


## 15. Hybrid recommender

In [ ]:
# CELL 42 — Hybrid CF + content recommender
def minmax_scale(values):
    values = np.asarray(values, dtype=np.float32)

    if len(values) == 0:
        return values

    lo, hi = values.min(), values.max()

    if hi - lo < 1e-8:
        return np.ones_like(values)

    return (values - lo) / (hi - lo)


def hybrid_recommendations(
    raw_user_id,
    n=10,
    candidate_n=100,
    cf_weight=0.85,
    content_weight=0.15
):
    raw_user_id = int(raw_user_id)

    if raw_user_id not in user_to_idx:
        raise ValueError("User is not in the training catalogue.")

    user_idx = user_to_idx[raw_user_id]

    candidate_indices, cf_scores = als_model.recommend(
        userid=user_idx,
        user_items=user_items[user_idx],
        N=candidate_n,
        filter_already_liked_items=True
    )

    candidate_indices = np.asarray(candidate_indices, dtype=np.int32)
    cf_scores = np.asarray(cf_scores, dtype=np.float32)

    user_profile = build_user_content_profile(raw_user_id)

    if user_profile is None:
        content_scores = np.zeros(len(candidate_indices), dtype=np.float32)
    else:
        content_scores = cosine_similarity(
            training_tfidf[candidate_indices],
            user_profile
        ).ravel().astype(np.float32)

    final_score = (
        cf_weight * minmax_scale(cf_scores)
        + content_weight * minmax_scale(content_scores)
    )

    result = pd.DataFrame({
        "movie_idx": candidate_indices,
        "cf_score": cf_scores,
        "content_score": content_scores,
        "final_score": final_score
    })

    result["movieId"] = result["movie_idx"].map(idx_to_movie)

    result = (
        result
        .merge(movies, on="movieId", how="left")
        .sort_values("final_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

    return result[
        ["movieId", "title", "genres", "cf_score", "content_score", "final_score"]
    ]

display(hybrid_recommendations(example_user, 10))


## 16. Recommendation diversity

In [ ]:
# CELL 43 — Simple genre diversification
def diversify_recommendations(recommendations, n=10, max_per_genre=3):
    selected_rows = []
    genre_counts = {}

    for _, row in recommendations.iterrows():
        genres = str(row.get("genres", ""))
        genre_list = [
            g for g in genres.split("|")
            if g and g != "(no genres listed)"
        ]

        allowed = all(
            genre_counts.get(g, 0) < max_per_genre
            for g in genre_list
        )

        if allowed:
            selected_rows.append(row)
            for g in genre_list:
                genre_counts[g] = genre_counts.get(g, 0) + 1

        if len(selected_rows) >= n:
            break

    return pd.DataFrame(selected_rows).reset_index(drop=True)

candidate_recs = hybrid_recommendations(
    example_user,
    n=50,
    candidate_n=150
)

display(diversify_recommendations(candidate_recs, n=10))


## 17. Cold start

In [ ]:
# CELL 44 — Cold-start user recommendations
def cold_start_recommendations(favorite_movie_ids, n=10):
    favorite_movie_ids = [
        int(mid) for mid in favorite_movie_ids
        if int(mid) in movie_to_idx
    ]

    if not favorite_movie_ids:
        return popularity_recommendations(n)

    favorite_indices = [
        movie_to_idx[mid] for mid in favorite_movie_ids
    ]

    user_profile = normalize(
        sp.csr_matrix(
            training_tfidf[favorite_indices].mean(axis=0)
        ),
        norm="l2"
    )

    scores = cosine_similarity(
        training_tfidf,
        user_profile
    ).ravel()

    result = pd.DataFrame({
        "movie_idx": np.arange(n_movies),
        "content_score": scores
    })

    result["movieId"] = result["movie_idx"].map(idx_to_movie)

    result = result[
        ~result["movieId"].isin(favorite_movie_ids)
    ]

    result = (
        result
        .merge(movies, on="movieId", how="left")
        .sort_values("content_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

    return result[
        ["movieId", "title", "genres", "content_score"]
    ]

display(cold_start_recommendations([liked_movie], 10))


## 18. Final recommendation router

In [ ]:
# CELL 45 — Final recommendation pipeline
def final_recommendation_pipeline(user_id, n=10):
    user_id = int(user_id)

    if user_id not in user_to_idx:
        result = popularity_recommendations(n).copy()
        result["recommendation_type"] = "cold_start"
        result["explanation"] = (
            "Popular movies shown because no interaction history exists."
        )
        return result

    candidates = hybrid_recommendations(
        user_id,
        n=50,
        candidate_n=150
    )

    result = diversify_recommendations(
        candidates,
        n=n,
        max_per_genre=3
    )

    result["recommendation_type"] = "personalized_hybrid"
    result["explanation"] = (
        "Personalized using collaborative filtering and movie-content signals."
    )

    return result

display(final_recommendation_pipeline(example_user, 10))


## 19. Optional MovieLens tags integration

In [ ]:
# CELL 46 — Load tags only now
tags = pd.read_csv(
    TAGS_PATH,
    usecols=["userId", "movieId", "tag"],
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "tag": "string"
    }
)

print("tags.shape:", tags.shape)
display(tags.head())


In [ ]:
# CELL 47 — Aggregate tags per movie
tags["tag"] = (
    tags["tag"]
    .fillna("")
    .str.lower()
    .str.strip()
)

movie_tags = (
    tags[tags["tag"] != ""]
    .groupby("movieId")["tag"]
    .agg(" ".join)
    .reset_index()
)

movie_metadata_full = movies.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movie_metadata_full["tag"] = movie_metadata_full["tag"].fillna("")
movie_metadata_full["genres_text"] = (
    movie_metadata_full["genres"]
    .fillna("")
    .str.replace("|", " ", regex=False)
)

movie_metadata_full["text"] = (
    movie_metadata_full["title"].fillna("")
    + " "
    + movie_metadata_full["genres_text"]
    + " "
    + movie_metadata_full["tag"]
)

display(movie_metadata_full.head())


In [ ]:
# CELL 48 — TF-IDF with title + genres + tags
TFIDF_TAG_MAX_FEATURES = 100000

tfidf_full = TfidfVectorizer(
    stop_words="english",
    max_features=TFIDF_TAG_MAX_FEATURES,
    min_df=2,
    ngram_range=(1, 2)
)

full_tfidf = tfidf_full.fit_transform(
    movie_metadata_full["text"]
)

print("TF-IDF with tags:", full_tfidf.shape)


In [ ]:
# CELL 49 — Release large tag dataframes
del tags
del movie_tags
gc.collect()

print("Tag dataframes released.")


## 20. Save model artifacts

In [ ]:
# CELL 50 — Save ALS model
joblib.dump(
    als_model,
    MODEL_DIR / "als_model.joblib",
    compress=3
)
print("Saved ALS model.")


In [ ]:
# CELL 51 — Save sparse matrices
sp.save_npz(MODEL_DIR / "user_items.npz", user_items)
sp.save_npz(MODEL_DIR / "user_movie_matrix.npz", user_movie_matrix)

print("Saved sparse matrices.")


In [ ]:
# CELL 52 — Save mappings and configuration
with open(MODEL_DIR / "user_to_idx.json", "w") as f:
    json.dump(user_to_idx, f)

with open(MODEL_DIR / "movie_to_idx.json", "w") as f:
    json.dump(movie_to_idx, f)

np.save(MODEL_DIR / "unique_users.npy", unique_users.to_numpy())
np.save(MODEL_DIR / "unique_movies.npy", unique_movies.to_numpy())

config = {
    "seed": SEED,
    "min_user_train": MIN_USER_TRAIN,
    "min_item_train": MIN_ITEM_TRAIN,
    "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
    "als_factors": ALS_FACTORS,
    "als_regularization": ALS_REGULARIZATION,
    "als_iterations": ALS_ITERATIONS,
    "tfidf_max_features": TFIDF_MAX_FEATURES
}

with open(MODEL_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=4)

print(json.dumps(config, indent=4))


In [ ]:
# CELL 53 — Save metadata and evaluation
movies.to_parquet(MODEL_DIR / "movies.parquet", index=False)

comparison.to_csv(
    OUTPUT_DIR / "baseline_vs_als.csv",
    index=False
)

pd.DataFrame([als_metrics]).to_csv(
    OUTPUT_DIR / "als_metrics.csv",
    index=False
)

joblib.dump(
    tfidf,
    MODEL_DIR / "tfidf.joblib",
    compress=3
)

sp.save_npz(
    MODEL_DIR / "training_tfidf.npz",
    training_tfidf
)

print("Metadata and evaluation artifacts saved.")


## 21. Reload test

In [ ]:
# CELL 54 — Reload saved artifacts
loaded_als_model = joblib.load(MODEL_DIR / "als_model.joblib")
loaded_user_items = sp.load_npz(MODEL_DIR / "user_items.npz")
loaded_movies = pd.read_parquet(MODEL_DIR / "movies.parquet")
loaded_tfidf = joblib.load(MODEL_DIR / "tfidf.joblib")
loaded_training_tfidf = sp.load_npz(MODEL_DIR / "training_tfidf.npz")

with open(MODEL_DIR / "config.json", "r") as f:
    loaded_config = json.load(f)

print("Model:", type(loaded_als_model))
print("User-item matrix:", loaded_user_items.shape)
print("Movie metadata:", loaded_movies.shape)
print("TF-IDF:", loaded_training_tfidf.shape)
print("Config:")
print(json.dumps(loaded_config, indent=2))


In [ ]:
# CELL 55 — Prediction using reloaded artifacts
def recommend_with_loaded_model(raw_user_id, n=10):
    raw_user_id = int(raw_user_id)

    if raw_user_id not in user_to_idx:
        return popularity_recommendations(n)

    user_idx = user_to_idx[raw_user_id]

    indices, scores = loaded_als_model.recommend(
        userid=user_idx,
        user_items=loaded_user_items[user_idx],
        N=n,
        filter_already_liked_items=True
    )

    result = pd.DataFrame({
        "movie_idx": indices.astype(int),
        "score": scores.astype(float)
    })

    result["movieId"] = result["movie_idx"].map(idx_to_movie)

    return (
        result
        .merge(loaded_movies, on="movieId", how="left")
        [["movieId", "title", "genres", "score"]]
        .reset_index(drop=True)
    )

display(recommend_with_loaded_model(example_user, 10))


## 22. Experiment tracking

In [ ]:
# CELL 56 — Record the current ALS experiment
experiment_record = pd.DataFrame([
    {
        "model": "ALS",
        "factors": ALS_FACTORS,
        "regularization": ALS_REGULARIZATION,
        "iterations": ALS_ITERATIONS,
        "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
        **als_metrics
    }
])

display(experiment_record)

experiment_record.to_csv(
    OUTPUT_DIR / "experiments.csv",
    index=False
)


## 23. Optional small hyperparameter experiment

Do this only after the baseline notebook works. Start with a few configurations and evaluate on a smaller user sample before committing to a larger search.

In [ ]:
# CELL 57 — ALS training helper
def train_als_model(
    user_items_matrix,
    factors=64,
    regularization=0.08,
    iterations=20
):
    model = AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=SEED
    )

    model.fit(
        user_items_matrix.T.tocsr(),
        show_progress=True
    )

    return model


In [ ]:
# CELL 58 — Small experiment grid
parameter_grid = [
    {"factors": 32, "regularization": 0.05, "iterations": 15},
    {"factors": 64, "regularization": 0.08, "iterations": 20},
    {"factors": 128, "regularization": 0.10, "iterations": 25},
]

experiment_results = []

for params in parameter_grid:
    print("=" * 80)
    print("Training:", params)

    start = time.time()

    model = train_als_model(
        user_items_matrix=user_items,
        factors=params["factors"],
        regularization=params["regularization"],
        iterations=params["iterations"]
    )

    metrics = evaluate_recommender(
        model=model,
        user_items_matrix=user_items,
        test_truth=test_truth,
        k=10,
        max_users=5000,
        seed=SEED
    )

    experiment_results.append({
        **params,
        **metrics,
        "train_minutes": (time.time() - start) / 60
    })

results_df = pd.DataFrame(experiment_results)

display(results_df.sort_values("NDCG@10", ascending=False))

results_df.to_csv(
    OUTPUT_DIR / "als_experiments.csv",
    index=False
)


## 24. API-style response

In [ ]:
# CELL 59 — Product/API output example
final_df = final_recommendation_pipeline(example_user, n=5)

api_response = {
    "user_id": int(example_user),
    "recommendations": []
}

for _, row in final_df.iterrows():
    score = row.get(
        "final_score",
        row.get("popularity_score", row.get("score", 0.0))
    )

    api_response["recommendations"].append({
        "movie_id": int(row["movieId"]),
        "title": str(row["title"]),
        "genres": str(row["genres"]),
        "score": float(score),
        "recommendation_type": str(
            row.get("recommendation_type", "personalized")
        ),
        "reason": str(
            row.get("explanation", "Recommended for you.")
        )
    })

print(json.dumps(api_response, indent=2))


# 25. Final ML Checklist

This notebook covers:

- MovieLens 32M loading
- Memory-efficient ratings ingestion
- Validation and EDA
- Chronological train/validation/test split
- Sparse User × Movie matrix
- Popularity baseline
- Item-Based Collaborative Filtering
- ALS Matrix Factorization
- Top-N recommendation
- Precision@K
- Recall@K
- HitRate@K
- NDCG@K 
- Coverage@K
- Similar movies
- Similar users
- User taste profile
- Explainable recommendations
- TF-IDF content model
- Hybrid CF + content recommendation
- Genre-based diversity
- Cold-start recommendations
- Optional 2M-row tags integration
- Model persistence
- Experiment tracking
- Reload validation
- API-style response

## Important

Do not tune dozens of configurations before the baseline works. First run through the notebook until **ALS training, recommendations, and evaluation** work; record the metrics, then continue to the hybrid model and experiments.
